# 外れ値検出(標準偏差 / z-score)

`formatted_trial.csv` の各数値列について z = (x - mean) / std を計算し、
|z| > しきい値 となる行を外れ値としてフラグします。

再現性のため、mean / std はサンプル標準偏差(ddof=1, pandasの既定)を使用し、乱数は使いません。

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 設定

In [11]:
###実験1

INPUT_PATH = "../data/formatted_trial_exp1.csv"   # 入力CSVのパス
THRESHOLD = 3.0                        # |z| > THRESHOLD を外れ値とする
OUTPUT_PATH = "../data/formatted_trial_flagged_exp1.csv"                    # None なら "<入力名>_flagged.csv" に自動設定

# 外れ値チェック対象の数値列(必要に応じて増減してください)
TARGET_COLUMNS = [
    "Page Submit_al",
    "Alertness Level",
    "Affect Level",
    "Page Submit_fc"
]

In [2]:
###実験2

INPUT_PATH = "../data/formatted_trial_exp2.csv"   # 入力CSVのパス
THRESHOLD = 3.0                        # |z| > THRESHOLD を外れ値とする
OUTPUT_PATH = "../data/formatted_trial_flagged_exp2.csv"                    # None なら "<入力名>_flagged.csv" に自動設定

# 外れ値チェック対象の数値列(必要に応じて増減してください)
TARGET_COLUMNS = [
    "Page Submit_wl",
    "Page Submit_al",
    "Water Level",
    "Alertness Level",
    "Affect Level",
    "Page Submit_fc"
]

## データ読み込み

In [12]:
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"以下の列がCSVに存在しません: {missing}")

print(df.shape)
df.head()

(8800, 9)


,responseId,frame_block,ref_subblock,Page Submit_al,Alertness Level,Affect Level,Page Submit_fc,frame_choice,Message Group
0,R_9nvFNaU5rMNP8Vi,0,0,5.553,28.0,39.0,8.803,1,neutral
1,R_9nvFNaU5rMNP8Vi,0,1,13.783,45.0,39.0,8.803,1,neutral
2,R_9nvFNaU5rMNP8Vi,0,2,4.873,38.0,49.0,8.803,1,neutral
3,R_9nvFNaU5rMNP8Vi,0,3,14.086,82.0,17.0,8.803,1,neutral
4,R_9nvFNaU5rMNP8Vi,0,4,13.431,60.0,48.0,8.803,1,neutral


## z-score の計算

In [13]:
def compute_zscores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """指定した列について z = (x - mean) / std を計算し、
    'z_<列名>' という列を追加した DataFrame を返す。"""
    out = df.copy()
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()  # ddof=1 (サンプル標準偏差)
        out[f"z_{col}"] = (df[col] - mean) / std
    return out


df_z = compute_zscores(df, TARGET_COLUMNS)
df_z[[f"z_{c}" for c in TARGET_COLUMNS]].describe()

,z_Page Submit_al,z_Alertness Level,z_Affect Level,z_Page Submit_fc
count,8.800000e+03,8.800000e+03,8.800000e+03,8.800000e+03
mean,3.229740e-17,1.195004e-16,-6.459479e-17,7.105427e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-6.851313e-01,-2.755349e+00,-2.448851e+00,-3.841374e-01
25%,-2.915440e-01,-6.170037e-01,-5.132960e-01,-2.914415e-01
50%,-1.134426e-01,-2.895878e-02,-2.940724e-02,-2.192380e-01
75%,1.237481e-01,5.056275e-01,5.996482e-01,-1.078613e-02
max,7.279891e+01,2.590514e+00,2.390037e+00,2.346859e+01


## 外れ値のフラグ付け

In [14]:
def flag_outliers(df_with_z: pd.DataFrame, columns: list[str], threshold: float) -> pd.DataFrame:
    """|z| > threshold となる行に is_outlier=True を付け、
    どの列が原因かを outlier_columns にまとめる。"""
    out = df_with_z.copy()
    z_cols = [f"z_{c}" for c in columns]

    is_outlier_per_col = out[z_cols].abs().gt(threshold)
    out["is_outlier"] = is_outlier_per_col.any(axis=1)
    out["outlier_columns"] = is_outlier_per_col.apply(
        lambda row: ", ".join(row.index[row].str.replace("z_", "", regex=False)),
        axis=1,
    )
    return out


df_flagged = flag_outliers(df_z, TARGET_COLUMNS, THRESHOLD)
df_flagged["is_outlier"].sum()

np.int64(108)

## サマリー

In [15]:
n_total = len(df_flagged)
print(f"閾値: |z| > {THRESHOLD}")
print(f"総行数: {n_total}\n")

print("=== 列ごとの外れ値件数 ===")
for col in TARGET_COLUMNS:
    n = (df_flagged[f"z_{col}"].abs() > THRESHOLD).sum()
    print(f"  {col:20s}: {n:5d} 件")

n_rows = int(df_flagged["is_outlier"].sum())
print(f"\n外れ値を1つ以上含む行: {n_rows} / {n_total} ({n_rows / n_total:.1%})")

閾値: |z| > 3.0
総行数: 8800

=== 列ごとの外れ値件数 ===
  Page Submit_al      :    30 件
  Alertness Level     :     0 件
  Affect Level        :     0 件
  Page Submit_fc      :    80 件

外れ値を1つ以上含む行: 108 / 8800 (1.2%)


In [16]:
if "responceId" in df_flagged.columns:
    by_resp = (
        df_flagged.groupby("responceId")["is_outlier"]
        .sum()
        .sort_values(ascending=False)
    )
    by_resp = by_resp[by_resp > 0]
    print(f"外れ値を1件以上持つ回答者: {len(by_resp)} 名\n")
    print("=== 回答者別 外れ値件数(上位10名) ===")
    print(by_resp.head(10).to_string())

## 外れ値の詳細確認

In [17]:
id_cols = [c for c in ["responceId", "frame_block", "ref_subblock"] if c in df_flagged.columns]
df_flagged.loc[df_flagged["is_outlier"], id_cols + TARGET_COLUMNS + ["outlier_columns"]]

,frame_block,ref_subblock,Page Submit_al,Alertness Level,Affect Level,Page Submit_fc,outlier_columns
832,6,2,89.138,50.0,40.0,3.866,Page Submit_al
903,0,3,70.232,45.0,45.0,20.740,Page Submit_al
1603,0,3,63.998,40.0,49.0,6.397,Page Submit_al
2165,13,0,189.903,38.0,60.0,3.249,Page Submit_al
2410,2,0,11.006,36.0,62.0,47.378,Page Submit_fc
...,...,...,...,...,...,...,...
8500,0,0,8.734,9.0,93.0,60.979,Page Submit_fc
8501,0,1,8.466,47.0,95.0,60.979,Page Submit_fc
8502,0,2,9.204,29.0,89.0,60.979,Page Submit_fc
8503,0,3,9.043,26.0,88.0,60.979,Page Submit_fc


## CSVへの書き出し

In [18]:
output_path = Path(OUTPUT_PATH) if OUTPUT_PATH else Path(INPUT_PATH).with_name(
    Path(INPUT_PATH).stem + "_flagged.csv"
)

keep_cols = list(df.columns) + ["is_outlier", "outlier_columns"]
df_flagged[keep_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"フラグ付きCSVを出力しました: {output_path}")

フラグ付きCSVを出力しました: ../data/formatted_trial_flagged_exp1.csv
